In [1]:
import os
import time
import requests
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

In [16]:
# Load API key from .env file
load_dotenv()
API_KEY = os.getenv("AQICN_API_KEY")

# API URL
url = f"https://api.waqi.info/feed/@10114/?token={API_KEY}"

In [17]:
# Function to fetch real-time AQI data with retries
def fetch_real_time_aqi():
    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=10)  # Set timeout
            data = response.json()

            if data.get("status") == "ok":
                # Extract pollutant data
                pollutants = data["data"]["iaqi"]

                # Convert to DataFrame and extract required pollutants
                df_real_time = pd.DataFrame({
                    "date": [datetime.today().strftime("%Y-%m-%d")],
                    "pm25": [pollutants.get("pm25", {}).get("v")],
                    "pm10": [pollutants.get("pm10", {}).get("v")],
                    "o3": [pollutants.get("o3", {}).get("v")],
                    "no2": [pollutants.get("no2", {}).get("v")],
                    "so2": [pollutants.get("so2", {}).get("v")],
                    "co": [pollutants.get("co", {}).get("v")],
                    "aqi": [data["data"]["aqi"]],
                })

                # Handle missing values using previous data or averages
                df_real_time.fillna(df_real_time.mean(numeric_only=True), inplace=True)

                # Save data to CSV
                file_path = "../data/real_time_aqi_data.csv"
                if os.path.exists(file_path):
                    df_real_time.to_csv(file_path, mode="a", header=False, index=False)
                else:
                    df_real_time.to_csv(file_path, mode="w", header=True, index=False)

                print("✅ Real-Time AQI Data Fetched Successfully:")
                print(df_real_time)
                return #df_real_time
            else:
                print(f"⚠️ API Error: {data.get('data')}")
        
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Attempt {attempt+1}/{max_retries} failed. Retrying...")
            time.sleep(5)

    print("❌ Failed to fetch AQI data after multiple attempts.")
    return None

In [18]:
# Run the function
fetch_real_time_aqi()

✅ Real-Time AQI Data Fetched Successfully:
         date  pm25  pm10    o3   no2  so2   co  aqi
0  2025-02-21   171   122  38.3  32.8  8.7  9.7  171
